In [1]:
from pathlib import Path
import difflib
import getpass
import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

SEED = 42
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

V1_DIR = PROJECT_ROOT / 'data' / 'clean' / 'en_uz' / 'public_5k_v1'
V2_DIR = PROJECT_ROOT / 'data' / 'clean' / 'en_uz' / 'public_5k_v2'
V2_DIR.mkdir(parents=True, exist_ok=True)

SCORED_PATH = V1_DIR / 'public_candidates_scored.csv'
OLD_AUDIT_PATH = V1_DIR / 'qwen_audit_results.csv'
LOCAL_SCORED_PATH = V2_DIR / 'candidates_with_local_quality.csv'
NEW_QUEUE_PATH = V2_DIR / 'new_llm_review_queue.csv'
NEW_AUDIT_PATH = V2_DIR / 'new_qwen_audit_results.csv'
NEW_USAGE_PATH = V2_DIR / 'new_qwen_audit_usage.json'

TARGET_PAIRS = 5000
NEW_QUEUE_SIZE = 3500
AUDIT_BATCH_SIZE = 25
AUDIT_MODEL = 'qwen-max'
DASHSCOPE_BASE_URL = 'https://dashscope.aliyuncs.com/compatible-mode/v1'
MIN_LABSE = 0.70
MIN_LLM_CONFIDENCE = 0.90

for required in (SCORED_PATH, OLD_AUDIT_PATH):
    if not required.exists():
        raise FileNotFoundError(f'缺少第一版结果: {required}')
print('第一版结果:', V1_DIR)
print('第二版输出:', V2_DIR)
print('新增 Qwen 审核目标:', NEW_QUEUE_SIZE, '条，约', int(np.ceil(NEW_QUEUE_SIZE / AUDIT_BATCH_SIZE)), '次请求')


第一版结果: D:\dev\projects\fourlang_translation\data\clean\en_uz\public_5k_v1
第二版输出: D:\dev\projects\fourlang_translation\data\clean\en_uz\public_5k_v2
新增 Qwen 审核目标: 3500 条，约 140 次请求


D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
WORD_RE = re.compile(r"[a-z]+(?:'[a-z]+)?")
SPACE_RE = re.compile(r'\s+')
REFERENCE_RE = re.compile(
    r'(?:^|\s)(retrieved|accessed|archived|official statistics|references?|bibliography|isbn|doi)(?:\s|$)'
    r'|[↑]|https?://|www\.|\[[0-9]{1,3}\](?:\[[0-9]{1,3}\])*',
    re.I,
)
EN_FUNCTION_WORDS = {
    'the', 'a', 'an', 'of', 'and', 'or', 'in', 'on', 'at', 'to', 'from', 'for', 'with',
    'without', 'by', 'as', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'that',
    'this', 'these', 'those', 'it', 'its', 'he', 'she', 'they', 'his', 'her', 'their',
    'who', 'which', 'after', 'before', 'during', 'between', 'into', 'also', 'but', 'not',
}

def clean_space(text):
    return SPACE_RE.sub(' ', str(text)).strip()

def local_quality_features(en, uz):
    en, uz = clean_space(en), clean_space(uz)
    en_tokens = WORD_RE.findall(en.lower())
    uz_tokens = WORD_RE.findall(uz.lower())
    uz_set = set(uz_tokens)
    token_overlap = sum(token in uz_set for token in en_tokens) / max(1, len(en_tokens))
    english_function_words_in_uz = sum(token in EN_FUNCTION_WORDS for token in uz_tokens)
    char_similarity = difflib.SequenceMatcher(None, en.lower(), uz.lower()).ratio()
    year_count = len(re.findall(r'\b(?:18|19|20)\d{2}\b', en))
    number_count = len(re.findall(r'\d+', en))

    reasons = []
    if REFERENCE_RE.search(en) or REFERENCE_RE.search(uz):
        reasons.append('reference_or_citation')
    if len(en_tokens) >= 6 and token_overlap >= 0.62:
        reasons.append('english_token_copy')
    if len(en) >= 35 and char_similarity >= 0.78:
        reasons.append('near_character_copy')
    if english_function_words_in_uz >= 2:
        reasons.append('english_function_words')
    if year_count >= 3 or number_count >= 8:
        reasons.append('table_or_list_fragment')
    if en.count('|') + uz.count('|') >= 2 or en.count('•') + uz.count('•') >= 2:
        reasons.append('structured_fragment')

    return {
        'token_overlap': round(token_overlap, 6),
        'english_function_words_in_uz': english_function_words_in_uz,
        'char_similarity': round(char_similarity, 6),
        'local_reject': bool(reasons),
        'local_reject_reason': '|'.join(reasons),
    }

scored_df = pd.read_csv(SCORED_PATH, keep_default_na=False)
old_audit_df = pd.read_csv(OLD_AUDIT_PATH, keep_default_na=False).drop_duplicates('pair_id', keep='last')
feature_df = pd.DataFrame([local_quality_features(en, uz) for en, uz in tqdm(zip(scored_df['en'], scored_df['uz']), total=len(scored_df), desc='本地二次质检')])
quality_df = pd.concat([scored_df.reset_index(drop=True), feature_df], axis=1)
quality_df.to_csv(LOCAL_SCORED_PATH, index=False, encoding='utf-8-sig')

old_merged = quality_df.merge(old_audit_df, on='pair_id', how='left', validate='one_to_one')
old_kept = old_merged[
    (~old_merged['local_reject']) &
    (old_merged['llm_label'] == 'correct') &
    (pd.to_numeric(old_merged['llm_confidence'], errors='coerce') >= MIN_LLM_CONFIDENCE) &
    (pd.to_numeric(old_merged['labse_score'], errors='coerce') >= MIN_LABSE)
].copy()

print('第一版严格合格且通过本地二次质检:', len(old_kept))
print('还需要补充约:', max(0, TARGET_PAIRS - len(old_kept)))
print('本地淘汰原因（允许一条包含多个原因）:')
print(quality_df.loc[quality_df['local_reject'], 'local_reject_reason'].value_counts().head(15))
print('保留来源分布:')
print(old_kept['source'].value_counts())


本地二次质检: 100%|██████████| 56563/56563 [00:11<00:00, 5120.48it/s]


第一版严格合格且通过本地二次质检: 2867
还需要补充约: 2133
本地淘汰原因（允许一条包含多个原因）:
local_reject_reason
reference_or_citation                                                                  7645
table_or_list_fragment                                                                 1088
reference_or_citation|english_token_copy|near_character_copy                            858
reference_or_citation|english_token_copy|near_character_copy|english_function_words     598
english_function_words                                                                  468
reference_or_citation|english_token_copy                                                429
english_token_copy|near_character_copy                                                  377
near_character_copy                                                                     350
reference_or_citation|table_or_list_fragment                                            210
english_token_copy|near_character_copy|english_function_words                           209
engl

In [3]:
audited_ids = set(old_audit_df['pair_id'].astype(str))
new_pool = quality_df[
    (~quality_df['local_reject']) &
    (~quality_df['pair_id'].astype(str).isin(audited_ids)) &
    (pd.to_numeric(quality_df['labse_score'], errors='coerce') >= MIN_LABSE)
].copy()
new_pool = new_pool.sort_values(['labse_score', 'pair_id'], ascending=[False, True])

non_wiki = new_pool[new_pool['source'] != 'wikimedia']
wiki = new_pool[new_pool['source'] == 'wikimedia']
new_queue = pd.concat([non_wiki, wiki.head(max(0, NEW_QUEUE_SIZE - len(non_wiki)))], ignore_index=True)
if len(new_queue) > NEW_QUEUE_SIZE:
    new_queue = new_queue.head(NEW_QUEUE_SIZE)
new_queue = new_queue.sort_values(['source', 'labse_score'], ascending=[True, False]).reset_index(drop=True)
new_queue.to_csv(NEW_QUEUE_PATH, index=False, encoding='utf-8-sig')

print('新增审核队列:', len(new_queue))
print(new_queue['source'].value_counts())
print(new_queue['labse_score'].describe(percentiles=[.01, .1, .5, .9, .99]))
display(new_queue.sample(min(12, len(new_queue)), random_state=SEED)[['en', 'uz', 'source', 'labse_score', 'token_overlap']])


新增审核队列: 3500
source
wikimedia     2925
tldr-pages     302
Tatoeba        273
Name: count, dtype: int64
count    3500.000000
mean        0.937376
std         0.031633
min         0.700931
1%          0.773570
10%         0.918390
50%         0.945976
90%         0.951325
99%         0.952589
max         0.952699
Name: labse_score, dtype: float64


,en,uz,source,labse_score,token_overlap
1650,"In 1975, he moved to Paris and became a produc...",1975-yilda u Parijga ko'chib o'tadi va France ...,wikimedia,0.948368,0.083333
2456,During the 2012 game he led his team to victor...,2012-yilgi o'yin davomida u jamoasini g'alabag...,wikimedia,0.945349,0.000000
2232,"Over the course of her 50-year career, Daniels...",50-yillik faoliyati davomida Daniels 230 ta fi...,wikimedia,0.946259,0.090909
1945,"Roger Keith Crouch (born September 12, 1940) i...",Roger Keith Crouch (1940-yil 12-sentyabrda tug...,wikimedia,0.947273,0.250000
309,This command is an alias of `docker container ...,Ushbu buyruq taxallus `docker container cp`.,tldr-pages,0.936717,0.333333
2341,Case Construction Equipment,Case Construction uskunalari,wikimedia,0.945813,0.666667
1666,"From 1855 to 1859 he lived in Paris, where he ...","1855-yildan 1859-yilgacha u Parijda yashab, u ...",wikimedia,0.948314,0.142857
1681,"It has been used, with minor changes, since 1972.",U 1972-yildan beri kichik oʻzgarishlar bilan i...,wikimedia,0.948258,0.000000
1187,The American Political Science Association (AP...,Amerika siyosiy fanlar assotsiatsiyasi ( APSA ...,wikimedia,0.950225,0.100000
325,This command is an alias of `docker container ...,Ushbu buyruq taxallus `docker container exec`.,tldr-pages,0.932169,0.333333


In [4]:
from openai import OpenAI

def extract_json(text):
    text = (text or '').strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.I)
        text = re.sub(r'\s*```$', '', text)
    start, end = text.find('{'), text.rfind('}')
    if start < 0 or end <= start:
        raise ValueError('响应中没有完整 JSON')
    return json.loads(text[start:end + 1])

def validate_results(payload, expected_ids):
    results = payload.get('results')
    if not isinstance(results, list):
        raise ValueError('JSON 缺少 results 数组')
    parsed = {}
    for item in results:
        pair_id = str(item.get('pair_id', ''))
        label = str(item.get('label', '')).lower()
        if pair_id not in expected_ids or label not in {'correct', 'minor', 'wrong', 'junk'}:
            continue
        try:
            confidence = min(1.0, max(0.0, float(item.get('confidence', 0))))
        except Exception:
            confidence = 0.0
        parsed[pair_id] = {
            'pair_id': pair_id, 'v2_label': label, 'v2_confidence': confidence,
            'v2_reason': clean_space(str(item.get('reason', '')))[:140],
        }
    missing = set(expected_ids) - set(parsed)
    if missing:
        raise ValueError(f'漏审 {len(missing)} 条')
    return [parsed[pair_id] for pair_id in expected_ids]

def audit_batch(client, batch_df, retries=4):
    records = batch_df[['pair_id', 'en', 'uz']].to_dict('records')
    prompt = f"""你是严格的英语-乌兹别克语平行语料质检员。这批数据将用于机器翻译训练。
逐条检查：乌兹别克语必须为自然的现代乌兹别克语拉丁字母，且完整表达英语含义。
必须特别拒绝：
1. 乌兹别克语列仍保留整句或大量英文，仅改日期、标点或少数字词；
2. 参考文献、网页标题堆叠、比赛表格、演员表、年份列表、导航碎片；
3. 数字、否定、专名、单位不一致；
4. 虽语义相似但不是自然完整句。
标签：correct=完整自然可直接训练；minor=轻微问题；wrong=错译/大量英文残留/内容不完整；junk=引用、列表、表格或噪声碎片。
只输出 JSON：{{"results":[{{"pair_id":"...","label":"correct|minor|wrong|junk","confidence":0.0,"reason":"correct留空，其余给极短原因"}}]}}
必须返回全部 {len(records)} 条，顺序不变。输入：{json.dumps(records, ensure_ascii=False)}"""
    last_error = None
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=AUDIT_MODEL, messages=[{'role': 'user', 'content': prompt}], temperature=0,
                response_format={'type': 'json_object'}, max_tokens=1900,
            )
            rows = validate_results(extract_json(response.choices[0].message.content), [r['pair_id'] for r in records])
            usage = getattr(response, 'usage', None)
            return rows, {
                'prompt_tokens': int(getattr(usage, 'prompt_tokens', 0) or 0),
                'completion_tokens': int(getattr(usage, 'completion_tokens', 0) or 0),
                'total_tokens': int(getattr(usage, 'total_tokens', 0) or 0),
            }
        except Exception as exc:
            last_error = exc
            if attempt + 1 < retries:
                time.sleep(2 ** attempt)
    raise RuntimeError(f'批量审核失败: {last_error}')

def atomic_csv_save(df, path):
    temp = path.with_suffix(path.suffix + '.tmp')
    df.to_csv(temp, index=False, encoding='utf-8-sig')
    temp.replace(path)

print('Qwen 只审核新增候选，不重复审核第一版已完成的数据。')
print('下一 Cell 默认不调用 API。')


Qwen 只审核新增候选，不重复审核第一版已完成的数据。
下一 Cell 默认不调用 API。


In [5]:
PREVIOUS_AUDIT_PATH = V2_DIR / 'new_qwen3_6_plus_audit_results.csv'
AUDIT_MODEL = 'deepseek-v4-flash'
AUDIT_BATCH_SIZE = 20
NEW_AUDIT_PATH = V2_DIR / 'new_mixed_audit_results.csv'
NEW_USAGE_PATH = V2_DIR / 'new_deepseek_v4_flash_resume_usage.json'
RESUME_QUEUE_SIZE = 4000
if len(new_queue) < RESUME_QUEUE_SIZE:
    new_queue = pd.concat([non_wiki, wiki.head(max(0, RESUME_QUEUE_SIZE - len(non_wiki)))], ignore_index=True)
    new_queue = new_queue.head(RESUME_QUEUE_SIZE).sort_values(['source', 'labse_score'], ascending=[True, False]).reset_index(drop=True)
    new_queue.to_csv(NEW_QUEUE_PATH, index=False, encoding='utf-8-sig')
    print('根据首批通过率，审核队列已扩展到:', len(new_queue))
print('本轮审核模型:', AUDIT_MODEL)

def audit_batch(client, batch_df, retries=3):
    records = batch_df[['pair_id', 'en', 'uz']].to_dict('records')
    expected_ids = [row['pair_id'] for row in records]
    prompt = '''请严格审核以下英语-乌兹别克语平行句。仅当乌兹别克语自然、完整翻译英语且没有大量英文残留时标 correct。引用、列表、表格、网页标题碎片标 junk；错译或大量英文残留标 wrong；轻微问题标 minor。只输出 JSON 对象，格式为 {"results":[{"pair_id":"...","label":"correct|minor|wrong|junk","confidence":0.0,"reason":""}]}，必须返回每个 pair_id。输入：''' + json.dumps(records, ensure_ascii=False)
    last_error = None
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=AUDIT_MODEL, messages=[{'role': 'user', 'content': prompt}],
                response_format={'type': 'json_object'}, max_tokens=2400, stream=False,
                extra_body={'thinking': {'type': 'disabled'}},
            )
            content = response.choices[0].message.content or ''
            if not content.strip():
                raise ValueError('DeepSeek 返回空内容')
            rows = validate_results(extract_json(content), expected_ids)
            usage = getattr(response, 'usage', None)
            return rows, {key: int(getattr(usage, key, 0) or 0) for key in ('prompt_tokens', 'completion_tokens', 'total_tokens')}
        except Exception as exc:
            last_error = exc
            if attempt + 1 < retries:
                time.sleep(2 ** attempt)
    if len(batch_df) > 1:
        middle = len(batch_df) // 2
        left_rows, left_usage = audit_batch(client, batch_df.iloc[:middle], retries=2)
        right_rows, right_usage = audit_batch(client, batch_df.iloc[middle:], retries=2)
        usage = {key: left_usage[key] + right_usage[key] for key in ('prompt_tokens', 'completion_tokens', 'total_tokens')}
        return left_rows + right_rows, usage
    raise RuntimeError(f'单条审核仍失败: {last_error}')
RUN_NEW_LLM_AUDIT = True

if RUN_NEW_LLM_AUDIT:
    api_key = getpass.getpass('请输入 DeepSeek 官方 API Key（不会保存，不能使用阿里云 DashScope Key）：')
    if not api_key.strip():
        raise ValueError('API Key 为空')
    client = OpenAI(api_key=api_key.strip(), base_url='https://api.deepseek.com')
    queue_df = pd.read_csv(NEW_QUEUE_PATH, keep_default_na=False)
    if NEW_AUDIT_PATH.exists():
        new_audit_df = pd.read_csv(NEW_AUDIT_PATH, keep_default_na=False)
    elif PREVIOUS_AUDIT_PATH.exists():
        new_audit_df = pd.read_csv(PREVIOUS_AUDIT_PATH, keep_default_na=False)
        new_audit_df['auditor_model'] = 'qwen3.6-plus'
        print('已导入 qwen3.6-plus 完成结果:', len(new_audit_df))
    else:
        new_audit_df = pd.DataFrame(columns=['pair_id', 'v2_label', 'v2_confidence', 'v2_reason', 'auditor_model'])
    if 'auditor_model' not in new_audit_df.columns:
        new_audit_df['auditor_model'] = 'qwen3.6-plus'
    new_audit_df = new_audit_df.drop_duplicates('pair_id', keep='last')
    done = set(new_audit_df['pair_id'].astype(str))
    pending = queue_df[~queue_df['pair_id'].astype(str).isin(done)].reset_index(drop=True)
    print('已完成:', len(done), '待审核:', len(pending), '支持断点续跑')

    usage_total = {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'completed_batches': 0}
    for start in tqdm(range(0, len(pending), AUDIT_BATCH_SIZE), desc='第二轮 Qwen 审核'):
        rows, usage = audit_batch(client, pending.iloc[start:start + AUDIT_BATCH_SIZE])
        for row in rows:
            row['auditor_model'] = AUDIT_MODEL
        new_audit_df = pd.concat([new_audit_df, pd.DataFrame(rows)], ignore_index=True).drop_duplicates('pair_id', keep='last')
        atomic_csv_save(new_audit_df, NEW_AUDIT_PATH)
        for key in ('prompt_tokens', 'completion_tokens', 'total_tokens'):
            usage_total[key] += usage[key]
        usage_total['completed_batches'] += 1
        NEW_USAGE_PATH.write_text(json.dumps(usage_total, ensure_ascii=False, indent=2), encoding='utf-8')

if NEW_AUDIT_PATH.exists():
    new_audit_df = pd.read_csv(NEW_AUDIT_PATH, keep_default_na=False)
    print('新增候选已审核:', new_audit_df['pair_id'].nunique(), '/', len(new_queue))
    display(new_audit_df['v2_label'].value_counts())
    if NEW_USAGE_PATH.exists():
        print('本次 Token:', json.loads(NEW_USAGE_PATH.read_text(encoding='utf-8')))
else:
    print('尚未调用 API；检查上一 Cell 样本后，将 RUN_NEW_LLM_AUDIT=True。')


根据首批通过率，审核队列已扩展到: 4000
本轮审核模型: deepseek-v4-flash
已导入 qwen3.6-plus 完成结果: 1000
已完成: 1000 待审核: 3000 支持断点续跑


第二轮 Qwen 审核: 100%|██████████| 150/150 [15:43<00:00,  6.29s/it]

新增候选已审核: 4000 / 4000


v2_label
correct    2767
minor       637
wrong       468
junk        128
Name: count, dtype: int64

本次 Token: {'prompt_tokens': 315265, 'completion_tokens': 140420, 'total_tokens': 455685, 'completed_batches': 150}


In [6]:
FINAL_PATH = V2_DIR / 'accepted_public_en_uz_5000_v2.csv'
TRAIN_PAIRS_PATH = V2_DIR / 'train_pairs_v2.csv'
VALID_PAIRS_PATH = V2_DIR / 'validation_pairs_v2.csv'
TRAIN_JSONL_PATH = V2_DIR / 'train_directed_v2.jsonl'
VALID_JSONL_PATH = V2_DIR / 'validation_directed_v2.jsonl'
HUMAN_REVIEW_PATH = V2_DIR / 'human_review_sample_v2.csv'
MANIFEST_PATH = V2_DIR / 'dataset_manifest_v2.json'

def write_directed(df, path):
    with path.open('w', encoding='utf-8') as f:
        for row in df.itertuples(index=False):
            common = {
                'pair_id': row.pair_id, 'source': row.source, 'source_version': row.source_version,
                'license': row.license, 'commercial_status': row.commercial_status,
                'labse_score': round(float(row.labse_score), 6),
            }
            for src_lang, tgt_lang, src_text, tgt_text in [
                ('en', 'uz', row.en, row.uz), ('uz', 'en', row.uz, row.en),
            ]:
                f.write(json.dumps({**common, 'src_lang': src_lang, 'tgt_lang': tgt_lang, 'src_text': src_text, 'tgt_text': tgt_text}, ensure_ascii=False) + '\n')

if not NEW_AUDIT_PATH.exists():
    print('请先完成新增候选审核，本 Cell 暂不导出。')
else:
    new_audit_df = pd.read_csv(NEW_AUDIT_PATH, keep_default_na=False).drop_duplicates('pair_id', keep='last')
    new_reviewed = new_queue.merge(new_audit_df, on='pair_id', how='left', validate='one_to_one')
    new_kept = new_reviewed[
        (new_reviewed['v2_label'] == 'correct') &
        (pd.to_numeric(new_reviewed['v2_confidence'], errors='coerce') >= MIN_LLM_CONFIDENCE) &
        (~new_reviewed['local_reject']) &
        (pd.to_numeric(new_reviewed['labse_score'], errors='coerce') >= MIN_LABSE)
    ].copy()
    new_kept['selection_stage'] = 'v2_new_audit'
    new_kept['final_llm_confidence'] = pd.to_numeric(new_kept['v2_confidence'], errors='coerce')
    old_kept_export = old_kept.copy()
    old_kept_export['selection_stage'] = 'v1_reused_after_local_filter'
    old_kept_export['final_llm_confidence'] = pd.to_numeric(old_kept_export['llm_confidence'], errors='coerce')

    combined = pd.concat([old_kept_export, new_kept], ignore_index=True, sort=False).drop_duplicates('pair_id')
    combined['final_rank'] = (
        pd.to_numeric(combined['labse_score'], errors='coerce') * 0.45 +
        combined['final_llm_confidence'] * 0.45 +
        (1 - pd.to_numeric(combined['token_overlap'], errors='coerce').clip(0, 1)) * 0.10
    )
    combined = combined.sort_values(['final_rank', 'pair_id'], ascending=[False, True]).head(TARGET_PAIRS).reset_index(drop=True)
    print('旧数据复用:', len(old_kept_export), '新增审核合格:', len(new_kept), '合计可用:', len(combined), '/', TARGET_PAIRS)
    if len(combined) < TARGET_PAIRS:
        print('尚未达到5000：不要放宽质量门槛。增大 NEW_QUEUE_SIZE 后重新运行新增队列和审核。')
    else:
        combined.to_csv(FINAL_PATH, index=False, encoding='utf-8-sig')
        rng = np.random.default_rng(SEED)
        order = rng.permutation(len(combined))
        valid_n = round(len(combined) * 0.05)
        valid_ids = set(order[:valid_n].tolist())
        valid_df = combined.iloc[sorted(valid_ids)].copy()
        train_df = combined.drop(index=sorted(valid_ids)).copy()
        train_df.to_csv(TRAIN_PAIRS_PATH, index=False, encoding='utf-8-sig')
        valid_df.to_csv(VALID_PAIRS_PATH, index=False, encoding='utf-8-sig')
        write_directed(train_df, TRAIN_JSONL_PATH)
        write_directed(valid_df, VALID_JSONL_PATH)

        review_parts = []
        for source, group in combined.groupby('source'):
            review_parts.append(group.sample(min(80, len(group)), random_state=SEED))
        high_overlap = combined.sort_values('token_overlap', ascending=False).head(100)
        review_df = pd.concat(review_parts + [high_overlap], ignore_index=True).drop_duplicates('pair_id').head(300)
        review_df.to_csv(HUMAN_REVIEW_PATH, index=False, encoding='utf-8-sig')

        manifest = {
            'dataset_name': 'public_en_uz_5k_v2', 'actual_pairs': len(combined),
            'train_pairs': len(train_df), 'validation_pairs': len(valid_df),
            'quality_controls': ['basic_filter', 'LaBSE', 'Qwen audit', 'English leakage filter', 'citation/list filter'],
            'source_counts': combined['source'].value_counts().to_dict(),
            'human_review_required_before_training': True,
            'warning': 'Automated quality filtering and public availability do not constitute legal clearance.',
        }
        MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
        print('第二版导出完成:', V2_DIR)
        print('训练方向记录:', len(train_df) * 2, '验证方向记录:', len(valid_df) * 2)
        print('最终来源分布:')
        print(combined['source'].value_counts())
        print('英文词重合率统计:')
        display(combined['token_overlap'].describe(percentiles=[.5, .75, .9, .95, .99]))
        display(combined.head(10)[['en', 'uz', 'source', 'labse_score', 'token_overlap', 'selection_stage']])


旧数据复用: 2867 新增审核合格: 2756 合计可用: 5000 / 5000
第二版导出完成: D:\dev\projects\fourlang_translation\data\clean\en_uz\public_5k_v2
训练方向记录: 9500 验证方向记录: 500
最终来源分布:
source
wikimedia     4756
Tatoeba        240
tldr-pages       4
Name: count, dtype: int64
英文词重合率统计:


count    5000.000000
mean        0.170210
std         0.142374
min         0.000000
50%         0.153846
75%         0.258065
90%         0.363636
95%         0.444444
99%         0.500185
max         0.666667
Name: token_overlap, dtype: float64

,en,uz,source,labse_score,token_overlap,selection_stage
0,Other statistics,Boshqa statistikalari,wikimedia,0.986685,0.0,v1_reused_after_local_filter
1,Other names,Boshqa nomlari,wikimedia,0.986515,0.0,v1_reused_after_local_filter
2,Other places,Boshqa joylar,wikimedia,0.984223,0.0,v1_reused_after_local_filter
3,He reigned from 1722 to 1735.,U 1722-yildan 1735-yilgacha hukmronlik qildi.,wikimedia,0.983687,0.0,v1_reused_after_local_filter
4,"7 medals (5 gold, 2 silver)","7 ta medal (5 ta oltin, 2 ta kumush)",wikimedia,0.983662,0.0,v1_reused_after_local_filter
5,"7 medals (3 gold, 2 silver, 2 bronze)","7 ta medal (3 ta oltin, 2 ta kumush, 2 ta bronza)",wikimedia,0.982920,0.0,v1_reused_after_local_filter
6,These include:,Bularga quyidagilar kiradi:,wikimedia,0.982745,0.0,v1_reused_after_local_filter
7,"15 medals (7 gold, 3 silver, 5 bronze)","15 ta medal (7 ta oltin, 3 ta kumush, 5 ta bro...",wikimedia,0.981604,0.0,v1_reused_after_local_filter
8,I don't understand you.,Sizni tushunmayman.,Tatoeba,0.980675,0.0,v1_reused_after_local_filter
9,Coupe de France: 2004–05,Fransiya kubogi: 2004–05,wikimedia,0.979996,0.0,v1_reused_after_local_filter
